# 09A · Do all-cell profiles recover clearer gene neighborhoods?

**Adagrasib mouse Visium HD.** Each point represents one gene. Start with Ada-14R and optionally repeat on Ada-9. The notebook compares all segmented cells in a mouse with T-cell-only and whole-tissue subsets matched by section and depth. It ranks predeclared marker pairs in the original expression profiles and in gene-PC space. It writes a human-readable pair comparison, figures, and an aggregate review ZIP.

## Settings

Place the extracted notebooks and Python modules together. Point `PROJECT_ROOT` to the CT26 project if the input adapter cannot resolve it from the current location. For an explicit full-tissue H5AD, supply its path in `SAMPLE_H5ADS`. Keep the same output root for 09A and 09B to reuse the prepared sample. The complete counts file stays on EC2.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
from input09 import prepare_sample
from review09 import show_review_zip
from genes09 import run_gene_pilot, review_gene_runs

SAMPLES = ["Ada-14R"]                       # Optional repeat: ["Ada-14R", "Ada-9"]
PROJECT_ROOT = None
SAMPLE_H5ADS = {}                          # Example: {"Ada-14R": "/path/to/full_tissue.h5ad"}
OUTPUT_ROOT = Path("09_pilot_results").expanduser().resolve()
CONFIG = dict(match_cells=30_000, max_genes=6_000, min_cells=12, n_pcs=35,
              neighbors=20, n_depth_bins=10, seed=17)
if len(SAMPLES) != len(set(SAMPLES)) or not SAMPLES:
    raise ValueError("Use one entry per selected mouse.")

## Prepare the full tissue and fit three views per mouse

The whole-tissue view uses all rows in the prepared sample. The two matched views draw the same number of cells in each physical section × library-depth stratum. One draw comes from the full tissue and one from the original T cell 1/2 labels. The draws may overlap. Both matched maps use the same gene universe. A gene with insufficient T-cell detection can still enter the full-tissue map and appears as unassessable for the matched T analysis.

Gene profiles use binary detection after subtracting each gene's section/depth-specific mean. A sparse randomized sketch followed by one streamed covariance refinement generates approximate gene PCs, followed by k-nearest neighbors, UMAP and Leiden. Read the method column if your environment uses a fallback. Each view fits its own coordinates; compare pair ranks and co-detection statistics across views.

In [ ]:
runs = []
for sample in SAMPLES:
    source = prepare_sample(sample, OUTPUT_ROOT, project_root=PROJECT_ROOT,
                            sample_h5ad=SAMPLE_H5ADS.get(sample))
    run_dir = run_gene_pilot(source, OUTPUT_ROOT, **CONFIG)
    runs.append(run_dir)
    return_zip, _, _ = review_gene_runs([run_dir], OUTPUT_ROOT)
    print("Return ZIP for this mouse:", return_zip)
    show_review_zip(return_zip)
    print(sample, "· completed gene maps", run_dir)
    display(pd.read_csv(run_dir / "view_sizes.csv"))
    display(Image(filename=str(run_dir / "gene_maps.png"), width=1250))
    display(Image(filename=str(run_dir / "gene_modules.png"), width=1250))

## Assess the positive controls and collect the return package

`direct_r` is depth- and section-adjusted within-cell co-detection. `PC_rank` is the partner's rank among the other mapped genes in higher-dimensional gene-PC space. `matched_null_percentile` compares the direct association to up to 100 non-panel genes of similar detection frequency in the same view. This descriptive comparison is not a permutation p-value. Review the cell-level readouts in 09B to assess changes after correction.

A reproducible gain for myeloid, endothelial, epithelial, B-cell and T-cell pairs supports use of the all-cell map for broad tissue programs. Whether CD4/CD8 assignment is supported within T cells is a separate question. The reported gains are within-mouse comparisons; the optional second mouse extends their assessment.

In [ ]:
review_zip, pairs, recurrence = review_gene_runs(runs, OUTPUT_ROOT)
display(pairs[["sample","panel","anchor","partner","direct_r_all_cells","direct_r_matched_all",
               "direct_r_matched_T","PC_rank_matched_all","PC_rank_matched_T",
               "matched_all_minus_T_r"]].round(3))
if len(runs) > 1:
    display(recurrence)
print("Return this 09A ZIP:", review_zip)
show_review_zip(review_zip)